In [3]:
# pip install opencv-python

In [4]:
# pip install ultralytics

In [1]:
# pip install lap

In [4]:
# =========================================================
# INSTALL REQUIREMENTS
# =========================================================
# افتح الترمينال في VSCode واكتب:
#
# pip install ultralytics supervision opencv-python numpy
#
# =========================================================
# IMPORTS
# =========================================================

import os
import cv2
import numpy as np

from collections import defaultdict, deque
from ultralytics import YOLO

# =========================================================
# PATHS
# =========================================================

VIDEO_PATH = r"../Video_inpt/Video_input_speed_violation_1.mp4"

OUTPUT_PATH = r"../video_out/output_speed_violation3copy.mp4"

VIOLATION_DIR = r"../VIOLATIONS_Frame/speed_violaiton_fram.jpg"

os.makedirs(VIOLATION_DIR, exist_ok=True)

# =========================================================
# LOAD MODELS
# =========================================================

# Vehicle Detection
vehicle_model = YOLO("../Models/yolo11m.pt")

# Road Segmentation
# road_model = YOLO("yolo11n-seg.pt")

# =========================================================
# VIDEO INFO
# =========================================================

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("FPS:", fps)
print("WIDTH:", width)
print("HEIGHT:", height)

# =========================================================
# VIDEO WRITER
# =========================================================

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)

# =========================================================
# SETTINGS
# =========================================================

# المسافة الحقيقية بين الخطين بالمتر
REAL_DISTANCE_METERS = 8

# السرعة القصوى
SPEED_LIMIT = 90

# # خطوط السرعة في Bird Eye
# LINE_UP = 390
# LINE_DOWN = 470
# خطوط السرعة في Bird Eye
LINE_UP = 750
LINE_DOWN = 850

# =========================================================
# BIRD EYE VIEW POINTS
# =========================================================

# SRC = np.float32([

#     [350, 260],   # top-left
#     [1500, 260],  # top-right
#     [1770, 1070], # bottom-right
#     [-390, 1070]  # bottom-left

# ])
SRC = np.float32([

    [500, 310],   # top-left
    [760, 300],  # top-right
    [2000, 1100], # bottom-right
    [-800, 1070]  # bottom-left

])

W = 1000
H = 1200

DST = np.float32([

    [0, 0],
    [W, 0],
    [W, H],
    [0, H]

])

# =========================================================
# PERSPECTIVE MATRIX
# =========================================================

matrix = cv2.getPerspectiveTransform(SRC, DST)

# =========================================================
# TRACKING VARIABLES
# =========================================================

track_history = defaultdict(
    lambda: deque(maxlen=30)
)

# تخزين frame ids
track_frame = {}

# السرعات النهائية
track_speed = {}

# العربيات المخالفة
violated_ids = set()

# =========================================================
# VEHICLE CLASSES
# =========================================================

# car, motorcycle, bus, truck
vehicle_classes = [2, 3, 5, 7]

# =========================================================
# ROAD MASK FUNCTION
# =========================================================

# def get_road_mask(frame):

#     results = road_model.predict(
#         frame,
#         conf=0.3,
#         verbose=False
#     )

#     mask = np.zeros(
#         (frame.shape[0], frame.shape[1]),
#         dtype=np.uint8
#     )

#     if results[0].masks is not None:

#         masks = results[0].masks.data.cpu().numpy()

#         for m in masks:

#             m = cv2.resize(
#                 m,
#                 (frame.shape[1], frame.shape[0])
#             )

#             mask[m > 0.5] = 255

#     return mask

# =========================================================
# CHECK INSIDE ROAD
# =========================================================

# def inside_road(cx, cy, road_mask):

#     if cx < 0 or cy < 0:
#         return False

#     if cy >= road_mask.shape[0]:
#         return False

#     if cx >= road_mask.shape[1]:
#         return False

#     return road_mask[cy, cx] > 0

# =========================================================
# TRANSFORM POINT
# =========================================================

def transform_point(x, y, matrix):

    point = np.array(
        [[[x, y]]],
        dtype=np.float32
    )

    transformed = cv2.perspectiveTransform(
        point,
        matrix
    )

    tx = int(transformed[0][0][0])

    ty = int(transformed[0][0][1])

    return tx, ty

# =========================================================
# MAIN LOOP
# =========================================================

frame_id = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_id += 1

    overlay = frame.copy()

    # =====================================================
    # DRAW SOURCE POLYGON
    # =====================================================

    # src_pts = SRC.astype(int)

    # cv2.polylines(
    #     overlay,
    #     [src_pts],
    #     True,
    #     (0,255,255),
    #     4
    # )

    # =====================================================
    # ROAD SEGMENTATION
    # =====================================================

    # road_mask = get_road_mask(frame)
    # # Save mask once
    # if frame_id == 1:

    #     mask_vis = cv2.cvtColor(
    #         road_mask,
    #         cv2.COLOR_GRAY2BGR
    #     )

    #     cv2.imwrite(
    #         "road_mask_color.jpg",
    #         mask_vis
    #     )

    # =====================================================
    # VISUALIZE ROAD MASK
    # =====================================================

    # green = np.zeros_like(frame)

    # green[:, :] = (0,255,0)

    # segmented = cv2.bitwise_and(
    #     green,
    #     green,
    #     mask=road_mask
    # )

    # overlay = cv2.addWeighted(
    #     overlay,
    #     1.0,
    #     segmented,
    #     0.25,
    #     0
    # )

    # =====================================================
    # BIRD EYE VIEW
    # =====================================================

    bird_eye = cv2.warpPerspective(
        frame,
        matrix,
        (W, H)
    )

    # =====================================================
    # DRAW BIRD EYE LINES
    # =====================================================

    cv2.line(
        bird_eye,
        (0, LINE_UP),
        (W, LINE_UP),
        (255,0,0),
        3
    )

    cv2.line(
        bird_eye,
        (0, LINE_DOWN),
        (W, LINE_DOWN),
        (0,0,255),
        3
    )

    # =====================================================
    # DETECTION + TRACKING
    # =====================================================

    results = vehicle_model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.35,
        iou=0.5,
        verbose=False
    )

    # =====================================================
    # CHECK IDS
    # =====================================================

    if results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()

        ids = results[0].boxes.id.cpu().numpy().astype(int)

        classes = results[0].boxes.cls.cpu().numpy().astype(int)

        # =================================================
        # LOOP OBJECTS
        # =================================================

        for box, track_id, cls in zip(
            boxes,
            ids,
            classes
        ):

            if cls not in vehicle_classes:
                continue

            x1, y1, x2, y2 = map(int, box)

            cx = int((x1 + x2) / 2)

            cy = int((y1 + y2) / 2)

            # =================================================
            # INSIDE ROAD
            # =================================================

            # if not inside_road(
            #     cx,
            #     cy,
            #     road_mask
            # ):
            #     continue

            # =================================================
            # TRANSFORM POINT
            # =================================================

            bx, by = transform_point(
                cx,
                cy,
                matrix
            )

            # =================================================
            # FILTER OUTSIDE BIRD VIEW
            # =================================================

            if bx < 0 or by < 0:
                continue

            if bx >= W or by >= H:
                continue

            # =================================================
            # TRACK HISTORY
            # =================================================

            track_history[track_id].append(
                (bx, by)
            )

            # =================================================
            # DIRECTION
            # =================================================

            direction = "UNKNOWN"

            if len(track_history[track_id]) >= 2:

                prev_y = track_history[
                    track_id
                ][-2][1]

                curr_y = track_history[
                    track_id
                ][-1][1]

                if curr_y > prev_y:
                    direction = "DOWN"
                else:
                    direction = "UP"

            # =================================================
            # SPEED CALCULATION USING FPS
            # =================================================

            if direction == "DOWN":

                # أول خط
                if track_id not in track_frame:

                    if abs(by - LINE_UP) < 10:

                        track_frame[
                            track_id
                        ] = frame_id

                # ثاني خط
                else:

                    if abs(by - LINE_DOWN) < 10:

                        frames_elapsed = (
                            frame_id
                            -
                            track_frame[track_id]
                        )

                        time_elapsed = (
                            frames_elapsed
                            / fps
                        )

                        if time_elapsed > 0:

                            speed = (
                                REAL_DISTANCE_METERS
                                /
                                time_elapsed
                            ) * 3.6

                            track_speed[
                                track_id
                            ] = int(speed)

                            del track_frame[
                                track_id
                            ]

            # =================================================
            # UP DIRECTION
            # =================================================

            elif direction == "UP":

                # أول خط
                if track_id not in track_frame:

                    if abs(by - LINE_DOWN) < 10:

                        track_frame[
                            track_id
                        ] = frame_id

                # ثاني خط
                else:

                    if abs(by - LINE_UP) < 10:

                        frames_elapsed = (
                            frame_id
                            -
                            track_frame[track_id]
                        )

                        time_elapsed = (
                            frames_elapsed
                            / fps
                        )

                        if time_elapsed > 0:

                            speed = (
                                REAL_DISTANCE_METERS
                                /
                                time_elapsed
                            ) * 3.6

                            track_speed[
                                track_id
                            ] = int(speed)

                            del track_frame[
                                track_id
                            ]

            # =================================================
            # CURRENT SPEED
            # =================================================

            current_speed = track_speed.get(
                track_id,
                0
            )

            # =================================================
            # VIOLATION
            # =================================================

            violation = (
                current_speed > SPEED_LIMIT
            )

            # =================================================
            # COLORS
            # =================================================

            color = (0,255,0)

            if violation:
                color = (0,0,255)

            # =================================================
            # DRAW BOX
            # =================================================

            cv2.rectangle(
                overlay,
                (x1, y1),
                (x2, y2),
                color,
                3
            )

            # =================================================
            # SPEED LABEL
            # =================================================

            if current_speed == 0:

                label = (
                    f"ID:{track_id} "
                    f"Calculating... "
                    f"{direction}"
                )

            else:

                label = (
                    f"ID:{track_id} "
                    f"{current_speed} km/h "
                    f"{direction}"
                )

            cv2.putText(
                overlay,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color,
                2
            )

            # =================================================
            # CENTER POINT
            # =================================================

            cv2.circle(
                overlay,
                (cx, cy),
                5,
                (255,255,0),
                -1
            )

            # =================================================
            # TRACK TRAIL
            # =================================================

            points = track_history[track_id]

            for i in range(1, len(points)):

                p1 = points[i-1]
                p2 = points[i]

                cv2.line(
                    bird_eye,
                    p1,
                    p2,
                    (255,0,255),
                    2
                )

            # =================================================
            # DRAW POINT IN BIRD VIEW
            # =================================================

            cv2.circle(
                bird_eye,
                (bx, by),
                5,
                color,
                -1
            )

            # =================================================
            # SAVE VIOLATION
            # =================================================
            




            if (
                violation
                and
                track_id not in violated_ids
            ):

                violated_ids.add(track_id)

            # Full frame

                violation_frame = overlay.copy()

                


                cv2.putText(
                    violation_frame,
                    f"Speed Limit: {SPEED_LIMIT} km/h",
                    (50, 70),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0,255,255),
                    3
                )


                cv2.putText(
                    violation_frame,
                    f"Vehicle ID: {track_id}",
                    (50,130),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0,0,255),
                    3
                )





                cv2.putText(
                    violation_frame,
                    f"Vehicle Speed: {current_speed} km/h",
                    (50, 180),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0,0,255),
                    3
                )

                cv2.putText(
                    violation_frame,
                    f"VIOLATION",
                    (50, 240),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0,0,255),
                    3
                )
                cv2.imwrite(
                os.path.join(
                    VIOLATION_DIR,
                    f"full_{track_id}_{current_speed}.jpg"
                ),
                violation_frame
                )

                # Vehicle crop
                crop = frame[y1:y2, x1:x2]

                



                

                if crop.size > 0:

                    cv2.imwrite(
                        os.path.join(
                            VIOLATION_DIR,
                            f"crop_{track_id}_{current_speed}.jpg"
                        ),
                        crop
                    )

                print(f"Violation Saved: ID {track_id}")












            # if (
            #     violation
            #     and
            #     track_id not in violated_ids
            # ):

            #     violated_ids.add(track_id)

            #     crop = frame[
            #         y1:y2,
            #         x1:x2
            #     ]

            #     if crop.size > 0:

            #         save_path = os.path.join(
            #             VIOLATION_DIR,
            #             f"car_{track_id}_{current_speed}.jpg"
            #         )

            #         cv2.imwrite(
            #             save_path,
            #             crop
            #         )

            #         print(
            #             f"Violation Saved: "
            #             f"{save_path}"
            #         )

    # =====================================================
    # SPEED LIMIT
    # =====================================================

    cv2.putText(
        overlay,
        f"Speed Limit: {SPEED_LIMIT} km/h",
        (50,80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0,255,255),
        3
    )

    # =====================================================
    # FRAME NUMBER
    # =====================================================

    cv2.putText(
        overlay,
        f"Frame: {frame_id}",
        (50,140),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255,255,255),
        3
    )

    # =====================================================
    # SHOW WINDOWS
    # =====================================================

    preview = cv2.resize(
        overlay,
        (1280, 720)
    )

    bird_preview = cv2.resize(
        bird_eye,
        (700, 900)
    )

    cv2.imshow("Speed Violation Detection", preview)

    cv2.imshow("Bird Eye View", bird_preview)

    # =====================================================
    # WRITE VIDEO
    # =====================================================

    out.write(overlay)

    # =====================================================
    # PRESS Q TO EXIT
    # =====================================================

    key = cv2.waitKey(1)

    if key == ord("q"):
        break

# =========================================================
# RELEASE
# =========================================================

cap.release()

out.release()

cv2.destroyAllWindows()

print("===================================")
print("DONE")
print("OUTPUT:", OUTPUT_PATH)
print("VIOLATIONS:", VIOLATION_DIR)
print("===================================")

FPS: 29.97002997002997
WIDTH: 1280
HEIGHT: 720
Violation Saved: ID 1
Violation Saved: ID 10
Violation Saved: ID 8
Violation Saved: ID 17
Violation Saved: ID 6
Violation Saved: ID 9
Violation Saved: ID 45
Violation Saved: ID 12
Violation Saved: ID 42
Violation Saved: ID 52
Violation Saved: ID 23
Violation Saved: ID 27
Violation Saved: ID 65
Violation Saved: ID 48
Violation Saved: ID 54
Violation Saved: ID 32
Violation Saved: ID 107
Violation Saved: ID 93
Violation Saved: ID 111
Violation Saved: ID 108
Violation Saved: ID 128
Violation Saved: ID 139
Violation Saved: ID 120
Violation Saved: ID 133
Violation Saved: ID 134
Violation Saved: ID 140
Violation Saved: ID 152
Violation Saved: ID 161
Violation Saved: ID 155
Violation Saved: ID 157
Violation Saved: ID 166
Violation Saved: ID 178
Violation Saved: ID 169
Violation Saved: ID 171
Violation Saved: ID 189
Violation Saved: ID 187
Violation Saved: ID 191
Violation Saved: ID 204
Violation Saved: ID 209
Violation Saved: ID 213
Violation Save